# ESM C Implementation (62% Train Spearman R, 43% Validation Spearman R)

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have limited chances to make queries.

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

import os
from tqdm.auto import tqdm

from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

## 1. Collect Training Data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [13]:
#with open('Hackathon_data/Hackathon_data/sequence.fasta', 'r') as f:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [14]:
len(sequence_wt)

656

In [15]:
def get_mutated_sequence(mut, sequence_wt):
    '''
    Adds the specified mutation into the wild-type sequence.

    Params:
        mut (str): The mutation to be applied.
        sequence_wt (str): The wild-type sequence to which the mutation will be applied.
    Returns:
        A deep copy of the mutated sequence string.
    '''
    # wt - wild type; pos - position; mt - mutation
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

    sequence = deepcopy(sequence_wt)

    return sequence[:pos] + mt + sequence[pos+1:]

In [16]:
#df_train = pd.read_csv('Hackathon_data/Hackathon_data/train.csv')
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [17]:
#df_test = pd.read_csv('Hackathon_data/Hackathon_data/test.csv')
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [18]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a Prediction Model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

### Hyperparameters

In [19]:
seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

### ProteinDataset
Convert amino acids to machine-readable data via embedding space used by ESM-2.

In [30]:
def gen_emb_from_df(df, out_dir="esm_c_embeddings_variants", device="cuda:0", batch_size=8):
    '''
    Generate and cache ESM-C sequence embeddings for each unique mutant.
    Saves one mean-pooled embedding tensor per mutant to out_dir/{mutant}.pt.
    '''
    os.makedirs(out_dir, exist_ok=True)

    if isinstance(device, torch.device):
        use_cuda = device.type == "cuda"
    else:
        use_cuda = str(device).startswith("cuda")
    device = torch.device(device) if not isinstance(device, torch.device) else device

    # Each item is (name_for_file, sequence)
    data = [(m, s[:1000]) for m, s in df[["mutant", "sequence"]].drop_duplicates().values]
    print(f"Number of unique variants: {len(data)}")

    # Instantiate 600-million-parameter ESM-C model
    model = ESMC.from_pretrained("esmc_600m").to(device).eval()
    if use_cuda:
        model.half()  # Half-prec. to reduce GPU memory and runtime

    for i in tqdm(range(int(np.ceil(len(data) / batch_size)))):
        batch = data[i * batch_size:(i + 1) * batch_size]

        # Cache skip
        if all(os.path.exists(os.path.join(out_dir, f"{name}.pt")) for name, _ in batch):
            continue

        for name, sequence in batch:
            path = os.path.join(out_dir, f"{name}.pt")
            if os.path.exists(path):
                continue

            protein = ESMProtein(sequence=sequence)

            with torch.no_grad():
                protein_tensor = model.encode(protein)
                model_output = model.logits(
                    protein_tensor,
                    LogitsConfig(sequence=True, return_embeddings=True),
                )

            seq_emb = model_output.embeddings

            # Handle both [L, D] and [1, L, D] output shapes.
            if seq_emb.ndim == 3:
                seq_emb = seq_emb[0]
            seq_mean = seq_emb.float().mean(dim=0).detach().cpu()

            torch.save(seq_mean, path)

class DmsESMDataset(Dataset):
    def __init__(self, df, emb_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mutant = self.df.loc[idx, "mutant"]
        emb = torch.load(os.path.join(self.emb_dir, f"{mutant}.pt")).float()
        if self.is_train:
            y = torch.tensor(self.df.loc[idx, "DMS_score"], dtype=torch.float32)
            return emb, y
        return emb, torch.tensor(0.0, dtype=torch.float32)

In [31]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No CUDA device found; using CPU.")

# Run on GPU 0 when available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [32]:
# Combine our datasets for embedding, them split them into train / test sets afterward. Each has mutated sequence variants stored as a separate .pt embedding.
all_df = pd.concat(
    [df_train[["mutant", "sequence"]], df_test[["mutant", "sequence"]]],
    ignore_index=True
).drop_duplicates("mutant")

gen_emb_from_df(all_df, out_dir="esm_c_embeddings_variants", device=device, batch_size=8)

Number of unique variants: 12464


  0%|          | 0/1558 [00:00<?, ?it/s]

In [33]:
# gen_emb('Hackathon_Data/Hackathon_data/sequence.fasta', out_dir='esm_embeddings_train')
# gen_emb('Hackathon_Data/Hackathon_data/', out_dir='esm_embeddings_test')

In [34]:
# Use simple MLP model to predict from ESM-2 embeddings.
class MLPRegressor(nn.Module):
    def __init__(self, input_dim=1280, hidden_dim=640, dropout_p=0.1):
        super(MLPRegressor, self).__init__()

        # Only need to predict a single fitness score every time.
        output_dim = 1

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid() # Ensures fitness scores in range (0,1).
        )

    def forward(self, x):
        return self.layers(x)

In [35]:
def train_model_esm(model, train_dataset, val_dataset, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Use MSE loss to handle bounded regression.
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_val_spearman = -np.inf
    best_ckpt = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(outputs.detach().cpu())
            train_targets.append(targets.detach().cpu())

        train_preds = torch.cat(train_preds).numpy()
        train_targets = torch.cat(train_targets).numpy()
        train_spearman = spearmanr(train_preds, train_targets).statistic

        # Validation
        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device).float()
                outputs = model(inputs).squeeze(-1)

                val_losses.append(criterion(outputs, targets).item())
                val_preds.append(outputs.detach().cpu())
                val_targets.append(targets.detach().cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_targets = torch.cat(val_targets).numpy()
        val_spearman = spearmanr(val_preds, val_targets).statistic
        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        if np.isnan(train_spearman):
            train_spearman = 0.0
        if np.isnan(val_spearman):
            val_spearman = -1.0

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss={mean_train_loss:.4f}, Train Spearman={train_spearman:.4f}, "
            f"Val Loss={mean_val_loss:.4f}, Val Spearman={val_spearman:.4f}"
        )

        # Early stopping on validation Spearman (ranking quality)
        if val_spearman > best_val_spearman:
            best_val_spearman = val_spearman
            best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_ckpt is None:
        best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return model, best_ckpt

In [36]:
# Split data into train and validation sets, then form DmsESMDataset() classes from each.
df_train_split, df_val_split = train_test_split(
    df_train, test_size=val_ratio, random_state=seed, shuffle=True
)

train_ds = DmsESMDataset(df_train_split, "esm_c_embeddings_variants", is_train=True)
val_ds = DmsESMDataset(df_val_split, "esm_c_embeddings_variants", is_train=True)
test_ds = DmsESMDataset(df_test, "esm_c_embeddings_variants", is_train=False)

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [37]:
# --------------- Train our model ---------------
model_esm = MLPRegressor(input_dim=1152,
                          hidden_dim=640,
                          dropout_p=0.3).to(device)
# TODO Currently using test values to make sure this works.
model_esm, best_ckpt_esm = train_model_esm(
    model_esm,
    train_ds,
    val_ds,
    epochs=150,  # 10
    batch_size=256,
    lr=1e-5,  # 1e-3
    patience=40,  # 5
    device=device,
 )
model_esm.load_state_dict(best_ckpt_esm)


# --------------- Test our model ---------------
model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader:
        # Inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        dms_score_pred = outputs.squeeze(-1).cpu().numpy()
        preds.extend(dms_score_pred.tolist())

df_pred = df_test.copy()
df_pred["DMS_score_predicted"] = preds

Epoch 1: Train Loss=0.1848, Train Spearman=0.1056, Val Loss=0.1197, Val Spearman=0.0986
Epoch 2: Train Loss=0.1810, Train Spearman=0.1427, Val Loss=0.1191, Val Spearman=0.0612
Epoch 3: Train Loss=0.1809, Train Spearman=0.1756, Val Loss=0.1191, Val Spearman=0.0908
Epoch 4: Train Loss=0.1753, Train Spearman=0.2206, Val Loss=0.1192, Val Spearman=0.0982
Epoch 5: Train Loss=0.1746, Train Spearman=0.2104, Val Loss=0.1194, Val Spearman=0.1581
Epoch 6: Train Loss=0.1696, Train Spearman=0.2699, Val Loss=0.1198, Val Spearman=0.1633
Epoch 7: Train Loss=0.1680, Train Spearman=0.2473, Val Loss=0.1205, Val Spearman=0.1936
Epoch 8: Train Loss=0.1629, Train Spearman=0.3304, Val Loss=0.1213, Val Spearman=0.2221
Epoch 9: Train Loss=0.1625, Train Spearman=0.3000, Val Loss=0.1223, Val Spearman=0.2314
Epoch 10: Train Loss=0.1611, Train Spearman=0.3081, Val Loss=0.1233, Val Spearman=0.2550
Epoch 11: Train Loss=0.1628, Train Spearman=0.3185, Val Loss=0.1244, Val Spearman=0.2705
Epoch 12: Train Loss=0.1550, T

In [38]:
# Show current ESM-based predictions.
df_pred[['mutant', 'DMS_score_predicted']].head(n=10)

,mutant,DMS_score_predicted
0,V1D,0.590740
1,V1Y,0.466763
2,V1C,0.355195
3,V1A,0.577860
4,V1E,0.578156
5,V1W,0.482111
6,V1T,0.630764
7,V1R,0.400403
8,V1Q,0.455701
9,V1S,0.613356


In [39]:
# Save predictions to .csv.
df_pred[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv', index=False)

In [40]:
group_number = "1"

with open("GroupName.txt", "w") as f:
    f.write(group_number.strip() + "\n")

print("Saved GroupName.txt")

Saved GroupName.txt


In [41]:
api_key = "df303a548bec1afcff7d7196650c5396cfa74c94a8be5357043602dcc7537ba7"

with open("APIKey.txt", "w") as f:
    f.write(api_key.strip() + "\n")

print("Saved APIKey.txt")

Saved APIKey.txt


## 3. Select Query for Next Round

In [50]:
# Show prediction distribution.
df_pred.sort_values('DMS_score_predicted', ascending=False)

,mutant,sequence,DMS_score_predicted
1362,V73A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.703209
9827,V577A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.697953
7142,V435E,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.690809
2393,Q132H,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.690103
9368,L553W,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.689753
...,...,...,...
3933,D233F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.119019
3943,D233V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.117568
3948,D233L,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.116053
4078,K240F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.108882


In [51]:
# Write mutants with top ten DMS scores to .txt file.
df_top_ten = df_pred.sort_values('DMS_score_predicted', ascending=False).head(n=10)
df_top_ten.to_csv('top10.txt', columns=['mutant'], index=False, header=False)
print(df_top_ten)

      mutant                                           sequence  \
1362    V73A  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
9827   V577A  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7142   V435E  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
2393   Q132H  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
9368   L553W  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7139   V435G  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
4445   V268A  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7390   K448P  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
10840  C630P  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
4232   F256I  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   

       DMS_score_predicted  
1362              0.703209  
9827              0.697953  
7142              0.690809  
2393              0.690103  
9368              0.689753  
7139              0.689510  
4445              0.686573  
7390              0.686073  
1

In [47]:
# Example: randomly select 100 test variants to be queried.
# Note: Random selection may not be a good strategy
# TODO: Select query mutants for the next round based on your own criteria

queries = np.random.choice(df_test.mutant.values, size=100, replace=False)
queries

array(['E284D', 'E415L', 'I562D', 'V433T', 'S502F', 'Y406M', 'V527E',
       'G6V', 'H95F', 'P354C', 'E274Q', 'L149S', 'E325C', 'N172L', 'R30L',
       'K269R', 'W348K', 'L75G', 'W348M', 'E532K', 'E469W', 'Q554A',
       'N74P', 'I279D', 'S439A', 'A174Y', 'V430E', 'C451K', 'R131F',
       'L627C', 'R237Q', 'V282E', 'A654D', 'L617Q', 'G55H', 'F171R',
       'F616D', 'L322D', 'P501S', 'V506L', 'Q484F', 'M337G', 'P354V',
       'V540S', 'P412L', 'V178C', 'W420M', 'V497S', 'L267S', 'Y437K',
       'P370L', 'R43G', 'M82I', 'Y403A', 'L539C', 'L350R', 'L14A',
       'E299T', 'S8K', 'A301V', 'N556P', 'Y188L', 'A194Q', 'D80M',
       'P571Q', 'R444A', 'G386T', 'E53F', 'V527H', 'S602V', 'C119G',
       'W338K', 'D37K', 'E567L', 'F238M', 'P412H', 'E3N', 'L36M', 'Y261H',
       'P394T', 'G139V', 'K104Y', 'Q146Y', 'S17F', 'E428K', 'K371A',
       'D628Q', 'T377E', 'E23M', 'G101Y', 'D244P', 'S426H', 'V175M',
       'F235T', 'P345C', 'F223K', 'T520F', 'K467Y', 'S303F', 'R343M'],
      dtype=object)

In [49]:
with open('query.txt', 'w') as f:
  for mutant in queries:
    f.write(mutant+'\n')